# Fully Self-Made OCR: Recognizer Training on Kaggle GPU (Tesla T4)

This notebook trains `TextRecognizerV2` (ResNet + 2-layer BiLSTM + CTC loss, 5.57M parameters) entirely from random initialisation. There are zero pretrained weights and zero ready-made OCR engines/APIs.

### End-to-End Workflow:
1. **Unpack & Verify**: Finds bundle under `/kaggle/input/` at any depth, copies to `/kaggle/working/ocr_project`, extracts the 4 `.tar` shards into `datasets/recognition_real/`, and strictly verifies all 53,397 crop images and manifest references.
2. **Environment & Audit**: Installs dependencies and runs non-fatal environment and project integrity checks.
3. **Isolated Smoke Test**: Runs a 50-step GPU smoke test isolated to `/tmp/ocr_smoke_test` (never touching `models/`).
4. **Failsafe Auto-Resume & Init Separation**:
   - `--init-from`: Transfers model weights ONLY from prior stage best checkpoint, resetting optimizer, scheduler, scaler, epochs (starting at 1), and `best_cer` (to `inf`).
   - `--resume`: Used strictly for mid-stage crash recovery to restore all state.
   - **Completion Markers**: Checks `stage_{stage}_completed.json` so reruns automatically skip completed stages.
   - **Checkpoint Pruning**: Keeps only the last 2 epoch checkpoints per stage plus the best checkpoint.
5. **Curriculum Stages**:
   - **Stage A**: 100% synthetic streaming data (30 epochs, lr=1e-3, 50,000 unique crops/epoch, early stopping on synthetic unseen-font val CER).
   - **Stage B**: Mixed synthetic + ALL 37,715 real FUNSD train crops. Sized 1:1 with streaming synthetic (37,715 fresh crops generated on the fly each epoch using training fonts only; total 75,430 samples/epoch; 25 epochs, lr=5e-4, early stopping on real FUNSD val CER).
   - **Stage C**: 100% Real FUNSD train crops only (the 37,715 real crops, excluding synthetic; 15 epochs, lr=1e-4, early stopping on real FUNSD val CER).
6. **Validation Evaluation**: Evaluates on real FUNSD val (greedy and beam search + LM).
   > **Strict Guard**: The held-out test split is NEVER evaluated in this notebook.
7. **Export**: Packages trained checkpoints and logs for one-click download.

In [ ]:
# Step 1: Detect bundle at any depth, unpack, extract tar shards, and verify 53,397 crops
import os
import sys
import json
import glob
import shutil
import tarfile
import zipfile
from pathlib import Path
import PIL

print(f"Python Version: {sys.version}")
print(f"Pillow Version: {PIL.__version__}")

WORKDIR = Path("/kaggle/working/ocr_project")
WORKDIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path("/kaggle/input")

print(f"\nScanning {INPUT_ROOT} for bundle at any depth...")

# 1. Search for .zip files
zip_candidates = list(INPUT_ROOT.rglob("*.zip"))
# 2. Search for extracted folders containing project marker files
extracted_candidates = [p.parent for p in INPUT_ROOT.rglob("pipeline.py")]

if zip_candidates:
    bundle_zip = zip_candidates[0]
    size_mb = bundle_zip.stat().st_size / (1024 * 1024)
    print(f"[FOUND ZIP] Located archive: {bundle_zip} ({size_mb:.2f} MB)")
    print(f"Extracting archive to {WORKDIR}...")
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall(WORKDIR)
    print("Archive extracted successfully!")
elif extracted_candidates:
    src_dir = extracted_candidates[0]
    print(f"[FOUND DIRECTORY] Located project folder: {src_dir}")
    print(f"Copying files from {src_dir} to {WORKDIR}...")
    for item in src_dir.iterdir():
        dest = WORKDIR / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)
    print("Copy completed successfully!")
else:
    raise FileNotFoundError(f"No .zip bundle or project directory found under {INPUT_ROOT}!")

os.chdir(WORKDIR)
print(f"Current working directory: {os.getcwd()}")

# 3. Extract the 4 .tar shards into datasets/
shards = sorted(list(WORKDIR.rglob("*.tar")))
print(f"\nFound {len(shards)} tar shard(s): {[s.name for s in shards]}")
if not shards:
    # Check if crops already exist on disk
    existing_crops = list((WORKDIR / "datasets" / "recognition_real").rglob("*.png"))
    if len(existing_crops) < 53397:
        raise FileNotFoundError(f"No .tar shards found and only {len(existing_crops)} crops exist on disk!")
else:
    target_extract_dir = WORKDIR / "datasets"
    target_extract_dir.mkdir(parents=True, exist_ok=True)
    for shard in shards:
        print(f"  Extracting {shard.name} ({shard.stat().st_size / (1024*1024):.2f} MB) into {target_extract_dir}...")
        with tarfile.open(shard, "r") as tf:
            if hasattr(tarfile, "data_filter"):
                tf.extractall(target_extract_dir, filter="data")
            else:
                tf.extractall(target_extract_dir)
    print("All tar shards extracted successfully!")

# 4. Rigorous verification: exactly 53,397 crop files and all manifest image paths resolve
real_crops_dir = WORKDIR / "datasets" / "recognition_real"
on_disk_crops = list(real_crops_dir.rglob("*.png"))
print(f"\nCrop files verified on disk in {real_crops_dir}: {len(on_disk_crops):,}")
if len(on_disk_crops) != 53397:
    raise ValueError(f"VERIFICATION FAILED: Expected exactly 53,397 crop files, but found {len(on_disk_crops):,}!")

missing_manifest_images = 0
total_manifest_images = 0
for m_name in ["recognition_train.jsonl", "recognition_val.jsonl", "recognition_test.jsonl"]:
    m_path = WORKDIR / "datasets" / m_name
    if not m_path.is_file():
        raise FileNotFoundError(f"Missing manifest: {m_path}")
    for line in m_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        img_rel = rec.get("image", "")
        if "recognition_real" in img_rel:
            total_manifest_images += 1
            img_full = WORKDIR / "datasets" / img_rel
            if not img_full.is_file():
                missing_manifest_images += 1

print(f"Verified {total_manifest_images:,} manifest image references. Missing: {missing_manifest_images}")
if missing_manifest_images > 0:
    raise FileNotFoundError(f"VERIFICATION FAILED: {missing_manifest_images} manifest image paths could not be found on disk!")

print("SUCCESS: Exactly 53,397 crops and all manifest references verified!")


In [ ]:
# Step 2: Install dependencies and run environment check (non-fatal)
!pip install --quiet pyclipper pypdfium2
!python check_env.py || echo "[NON-FATAL] check_env reported warnings; continuing."


In [ ]:
# Step 3: Run project audit and unit tests (non-fatal)
!python check_project.py || echo "[NON-FATAL] check_project reported warnings; continuing."
!python -m unittest tests/test_recognizer_v2.py || echo "[NON-FATAL] unittest reported test warnings; continuing."


In [ ]:
# Step 4: Run Isolated 50-Step Smoke Test
# Must write to /tmp/ocr_smoke_test so models/ is NEVER touched or polluted
!python train_recognizer_gpu.py \
    --stage A \
    --epochs 1 \
    --max-steps 50 \
    --batch-size 32 \
    --output-dir /tmp/ocr_smoke_test \
    --amp


In [ ]:
# Step 5: Curriculum Stage Setup & Resume/Init Scanner
# Differentiates --init-from (new stage weights transfer) from --resume (crash recovery)
# Checks completion markers to skip already completed stages.
import glob
import os
import re

def get_stage_args(stage):
    """
    Determines training flags for the given curriculum stage.
    1. If stage completion marker exists, returns None (skip).
    2. If epoch checkpoints for this stage exist, returns '--resume <latest_epoch>' for crash recovery.
    3. If starting stage fresh:
       - Stage A: returns '' (random initialization)
       - Stage B: requires Stage A best and returns '--init-from models/recognizer_stageA_best.pt'
       - Stage C: requires Stage B best and returns '--init-from models/recognizer_stageB_best.pt'
    """
    marker = f"models/stage_{stage}_completed.json"
    if os.path.isfile(marker):
        print(f"[STAGE COMPLETED] Found marker: {marker}. Skipping Stage {stage}.")
        return None

    stage_epochs = glob.glob(f"models/recognizer_stage{stage}_epoch*.pt")
    if stage_epochs:
        def parse_ep(p):
            m = re.search(r"epoch(\d+)", p)
            return int(m.group(1)) if m else 0
        latest_epoch_ckpt = sorted(stage_epochs, key=parse_ep)[-1]
        print(f"[RESUME] Found existing {stage} epoch checkpoint: {latest_epoch_ckpt}")
        return f"--resume {latest_epoch_ckpt}"

    if stage == "A":
        print("[INITIALIZE] Starting Stage A from random initialization.")
        return ""
    elif stage == "B":
        prior_best = "models/recognizer_stageA_best.pt"
        if not os.path.isfile(prior_best):
            raise FileNotFoundError(
                f"CANNOT RUN STAGE B: Required checkpoint '{prior_best}' not found!\n"
                f"Stage A must complete and produce a best checkpoint before Stage B can start."
            )
        print(f"[INITIALIZE] Initializing Stage B weights from Stage A best: {prior_best}")
        return f"--init-from {prior_best}"
    elif stage == "C":
        prior_best = "models/recognizer_stageB_best.pt"
        if not os.path.isfile(prior_best):
            raise FileNotFoundError(
                f"CANNOT RUN STAGE C: Required checkpoint '{prior_best}' not found!\n"
                f"Stage B must complete and produce a best checkpoint before Stage C can start."
            )
        print(f"[INITIALIZE] Initializing Stage C weights from Stage B best: {prior_best}")
        return f"--init-from {prior_best}"

print("Stage scanner verified. Existing models directory checkpoints:", glob.glob("models/*.pt"))


In [ ]:
# Step 6: Curriculum Stage A (100% Synthetic Streaming Training, 30 epochs, lr=1e-3)
# Early stopping on synthetic val CER (unseen fonts)
stage_a_args = get_stage_args("A")
if stage_a_args is not None:
    !python train_recognizer_gpu.py \
        --stage A \
        --epochs 30 \
        --batch-size 64 \
        --lr 1e-3 \
        --synth-count 50000 \
        --amp {stage_a_args}
else:
    print("[SKIP] Stage A already completed. Proceeding to Stage B.")


In [ ]:
# Step 7: Curriculum Stage B (50/50 Streaming Synthetic + ALL 37,715 Real FUNSD crops, 25 epochs, lr=5e-4)
# Automatically derives 37,715 fresh streaming synthetic samples/epoch matching real crops (total 75,430)
# Early stopping strictly on Real FUNSD val CER
stage_b_args = get_stage_args("B")
if stage_b_args is not None:
    !python train_recognizer_gpu.py \
        --stage B \
        --epochs 25 \
        --batch-size 64 \
        --lr 5e-4 \
        --amp {stage_b_args}
else:
    print("[SKIP] Stage B already completed. Proceeding to Stage C.")


In [ ]:
# Step 8: Curriculum Stage C (100% Real FUNSD Train crops only, 37,715 samples, 15 epochs, lr=1e-4)
# Early stopping strictly on Real FUNSD val CER
stage_c_args = get_stage_args("C")
if stage_c_args is not None:
    !python train_recognizer_gpu.py \
        --stage C \
        --epochs 15 \
        --batch-size 64 \
        --lr 1e-4 \
        --amp {stage_c_args}
else:
    print("[SKIP] Stage C already completed. Proceeding to evaluation.")


In [ ]:
# Step 9: Validation Evaluation (Greedy & Beam Search on Real Validation Split ONLY)
# NOTE: The test split is STRICTLY NOT evaluated in this notebook to protect test data integrity.
best_c = "models/recognizer_stageC_best.pt"

print("=" * 75)
print("Evaluating Best Stage C Recognizer on Real FUNSD Validation Split (Greedy CTC)")
print("=" * 75)
!python evaluate_recognizer.py \
    --checkpoint {best_c} \
    --split val \
    --output outputs/val_metrics_greedy.json

print("\n" + "=" * 75)
print("Evaluating Best Stage C Recognizer on Real FUNSD Validation Split (Beam Search + LM)")
print("=" * 75)
!python evaluate_recognizer.py \
    --checkpoint {best_c} \
    --split val \
    --beam-width 10 \
    --lm-weight 0.4 \
    --output outputs/val_metrics_beam.json


In [ ]:
# Step 10: Export Checkpoints and Logs for Download
import os
import zipfile
from pathlib import Path

export_zip = "/kaggle/working/recognizer_training_outputs.zip"
print(f"Exporting models/ and outputs/ to {export_zip}...")

with zipfile.ZipFile(export_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in Path("models").glob("recognizer_*.pt"):
        zf.write(p, arcname=f"models/{p.name}")
    for p in Path("models").glob("stage_*_completed.json"):
        zf.write(p, arcname=f"models/{p.name}")
    for p in Path("outputs").glob("recognizer_train_log.*"):
        zf.write(p, arcname=f"outputs/{p.name}")
    for p in Path("outputs").glob("val_metrics_*.json"):
        zf.write(p, arcname=f"outputs/{p.name}")

size_mb = os.path.getsize(export_zip) / (1024 * 1024)
print(f"Export complete! Archive size: {size_mb:.2f} MB")
print("Download recognizer_training_outputs.zip from the Output tab on the right panel.")
